In [9]:
"""
Conv1 검증 스크립트 (수정본)
  - .npy 파일로 Python reference 계산
  - 시뮬레이션 출력(conv1_out.hex)과 비교

수정 내역 (기존 코드 대비):
  1. conv1_forward: >>10 arithmetic right shift 추가 (핵심 수정)
     이전: mac_sum → saturate [-128,127] 바로 적용 (틀림)
     수정: mac_sum → >>10 → saturate [-128,127] → ReLU [0,127]

  2. conv1_forward: ReLU 추가
     이전: ReLU 없음 → d0(-48) 같은 음수 출력 가능
     수정: >>10 + saturate 후 max(0, ...) 적용

  3. load_weights 제거
     .mem 파일 대신 .npy 파일 직접 사용 (언패킹 오류 원천 제거)
     weight shape: (8, 1, 3, 3) int8

  4. 하드웨어 비트 정확 모사 (bit-true):
     - int32 누적 → arithmetic >>10 → saturate → ReLU
     - Python의 >> 연산자는 음수에서도 arithmetic shift ✓
"""

import numpy as np

# ============================================================
# 경로 설정 (본인 환경에 맞게 수정)
# ============================================================
NPY_DIR = r"C:\Users\111eh\AppData\Local\Temp\BNZ.6a153c8824bc5259"
SIM_HEX = r"C:\Users\111eh\INTELLIGENT_SYSTEM_DESIGN\Assignment4\Assignment4_top\Assignment4_top.sim\sim_1\behav\xsim\conv1_out.hex"
IMG_IDX = 0   # 검증할 이미지 인덱스 (0~9999)

# ============================================================
# 1. .npy 로드
# ============================================================
weight = np.load(f"{NPY_DIR}/layer1_0_weight.npy")  # (8, 1, 3, 3) int8
inp    = np.load(f"{NPY_DIR}/input.npy")            # (10000, 1, 28, 28) int8

img = inp[IMG_IDX, 0].astype(np.int32)  # (28, 28) int32로 변환 (누적 오버플로 방지)

print(f"이미지  shape: {inp.shape},  dtype: {inp.dtype}")
print(f"가중치  shape: {weight.shape}, dtype: {weight.dtype}")
print(f"검증 이미지 인덱스: {IMG_IDX}")
print(f"img  min={img.min()}, max={img.max()}")
print(f"wt   min={weight.min()}, max={weight.max()}")

OC, IC, KH, KW = weight.shape  # 8, 1, 3, 3
OUT_H, OUT_W   = 26, 26

# ============================================================
# 2. 하드웨어 bit-true Convolution
#
#    하드웨어 처리 순서 (과제 스펙):
#      (a) int8 × int8 곱셈 → int32 누적 (9회 MAC, IC=1)
#      (b) arithmetic right shift >>> 10  (LSB 10비트 버림)
#      (c) saturate: 값 > 127 → 127, 값 < -128 → -128
#      (d) ReLU: 값 < 0 → 0
#
#    Python >> 는 음수에서 arithmetic shift 동작 ✓
#    따라서 별도 부호 처리 불필요
# ============================================================
ref = np.zeros((OC, OUT_H, OUT_W), dtype=np.int32)

for oc in range(OC):
    for r in range(OUT_H):
        for c in range(OUT_W):
            # (a) MAC 누적 (IC=1이므로 단순 sum)
            window = img[r:r+KH, c:c+KW]                       # (3,3) int32
            wt     = weight[oc, 0].astype(np.int32)             # (3,3) int32
            acc    = int(np.sum(window * wt))                   # scalar int

            # (b) arithmetic right shift 10
            shifted = acc >> 10

            # (c) saturate [-128, 127]
            saturated = max(-128, min(127, shifted))

            # (d) ReLU
            ref[oc, r, c] = max(0, saturated)

ref = ref.astype(np.int8)
print(f"\n[Python ref] shape={ref.shape}, min={ref.min()}, max={ref.max()}")

# ============================================================
# 3. 시뮬레이션 결과 로드
#    conv1_out.hex 포맷: 채널별 순서, 각 값 2자리 hex (8비트 unsigned)
#    예) 0x00=0, 0x7F=127, 0xFF=-1(signed), 0xD0=208(unsigned)=-48(signed)
# ============================================================
raw_list = []
with open(SIM_HEX, "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("//"):
            continue
        val = int(line, 16) & 0xFF          # 8비트 마스킹
        # unsigned → signed
        raw_list.append(val if val < 128 else val - 256)

raw = np.array(raw_list, dtype=np.int8).reshape(OC, OUT_H, OUT_W)
print(f"[Sim]       shape={raw.shape}, min={raw.min()}, max={raw.max()}")

# ============================================================
# 4. 비교
# ============================================================
match = np.array_equal(ref, raw)
diff  = ref.astype(np.int32) - raw.astype(np.int32)

print(f"\n=== 결과 {'✅ 일치' if match else '❌ 불일치'} ===")
print(f"  최대 오차:      {np.abs(diff).max()}")
print(f"  불일치 픽셀 수: {np.sum(diff != 0)} / {OC * OUT_H * OUT_W}")

if not match:
    idx = np.argwhere(diff != 0)
    print("\n  [처음 10개 불일치]")
    print(f"  {'ch':>3} {'row':>4} {'col':>4}  {'ref':>5}  {'sim':>5}  {'diff':>5}")
    for i in idx[:100]:
        oc, r, c = i
        print(f"  {oc:3d} {r:4d} {c:4d}  {ref[oc,r,c]:5d}  {raw[oc,r,c]:5d}  {diff[oc,r,c]:5d}")

    # 채널별 불일치 통계
    print("\n  [채널별 불일치 수]")
    for oc in range(OC):
        n = np.sum(diff[oc] != 0)
        if n > 0:
            print(f"  ch{oc}: {n} 픽셀 불일치")

# ============================================================
# 5. (선택) 정답 hex 파일 저장
#    시뮬레이션 비교용으로 python_conv1_ref.hex 생성
# ============================================================
with open("python_conv1_ref.hex", "w") as f:
    for oc in range(OC):
        for r in range(OUT_H):
            for c in range(OUT_W):
                val = int(ref[oc, r, c])
                f.write(f"{val & 0xFF:02x}\n")

print("\n[저장] python_conv1_ref.hex 생성 완료 (5408줄)")

이미지  shape: (10000, 1, 28, 28),  dtype: int8
가중치  shape: (8, 1, 3, 3), dtype: int8
검증 이미지 인덱스: 0
img  min=0, max=127
wt   min=-127, max=123

[Python ref] shape=(8, 26, 26), min=0, max=57
[Sim]       shape=(8, 26, 26), min=0, max=57

=== 결과 ❌ 불일치 ===
  최대 오차:      35
  불일치 픽셀 수: 420 / 5408

  [처음 10개 불일치]
   ch  row  col    ref    sim   diff
    0    3   24      0      4     -4
    0    3   25      0      4     -4
    1    3   24      0      6     -6
    1    3   25      0      6     -6
    1    4   24      0      7     -7
    1    4   25      0      7     -7
    3    3   24      0      4     -4
    3    3   25      0      4     -4
    3    4   24      0      1     -1
    3    4   25      0      1     -1
    4    3   18      2      0      2
    4    3   19      0      2     -2
    4    3   23      1      0      1
    4    5    6      2      0      2
    4    5    7      0      2     -2
    4    6    5      2      0      2
    4    6    6      9      2      7
    4    6    7      0      

In [14]:
"""
Conv1 검증 스크립트
  - Python으로 conv1 결과 계산 (reference)
  - 시뮬레이션 출력(conv1_out.hex) 과 비교
"""

import numpy as np

NPY_DIR = r"C:\Users\111eh\AppData\Local\Temp\BNZ.6a153c8824bc5259"
SIM_HEX = r"C:\Users\111eh\INTELLIGENT_SYSTEM_DESIGN\Assignment4\Assignment4_top\Assignment4_top.sim\sim_1\behav\xsim\conv1_out.hex"
IMG_IDX = 0

# ── 1. Python reference 계산 ──────────────────────────────────────────────────
weight = np.load(f"{NPY_DIR}/layer1_0_weight.npy")   # (8, 1, 3, 3) int8
inp    = np.load(f"{NPY_DIR}/input.npy")              # (10000,1,28,28) int8
img    = inp[IMG_IDX, 0].astype(np.int32)             # (28,28)

OC, IC, KH, KW = weight.shape   # 8,1,3,3
OUT_H, OUT_W = 26, 26

ref = np.zeros((OC, OUT_H, OUT_W), dtype=np.int32)
for oc in range(OC):
    for r in range(OUT_H):
        for c in range(OUT_W):
            acc = 0
            for kr in range(KH):
                for kc in range(KW):
                    acc += int(img[r+kr, c+kc]) * int(weight[oc, 0, kr, kc])
            # arithmetic right shift 10 + ReLU + saturate [0,127]
            shifted = acc >> 10
            ref[oc, r, c] = max(0, min(127, shifted))

ref = ref.astype(np.int8)
print(f"[Python] ref shape: {ref.shape}, min={ref.min()}, max={ref.max()}")

# ── 2. 시뮬레이션 결과 로드 ───────────────────────────────────────────────────
raw = []
with open(SIM_HEX, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            val = int(line, 16)
            # 8-bit unsigned → signed
            raw.append(val if val < 128 else val - 256)

raw = np.array(raw, dtype=np.int8).reshape(8, 26, 26)
print(f"[Sim]    sim shape: {raw.shape}, min={raw.min()}, max={raw.max()}")

# ── 3. 비교 ───────────────────────────────────────────────────────────────────
match = np.array_equal(ref, raw)
diff  = ref.astype(np.int32) - raw.astype(np.int32)

print(f"\n=== 결과 {'✅ 일치' if match else '❌ 불일치'} ===")
print(f"  최대 오차: {np.abs(diff).max()}")
print(f"  불일치 픽셀 수: {np.sum(diff != 0)} / {8*26*26}")

if not match:
    idx = np.argwhere(diff != 0)
    print("\n  [처음 5개 불일치]")
    print(f"  {'ch':>3} {'row':>4} {'col':>4}  {'ref':>5}  {'sim':>5}  {'diff':>5}")
    for i in idx[:5]:
        oc, r, c = i
        print(f"  {oc:3d} {r:4d} {c:4d}  {ref[oc,r,c]:5d}  {raw[oc,r,c]:5d}  {diff[oc,r,c]:5d}")


[Python] ref shape: (8, 26, 26), min=0, max=57
[Sim]    sim shape: (8, 26, 26), min=0, max=57

=== 결과 ✅ 일치 ===
  최대 오차: 0
  불일치 픽셀 수: 0 / 5408


In [15]:
"""
Conv1 검증 스크립트
  - Python으로 conv1 결과 계산 (reference)
  - 시뮬레이션 출력(conv1_out.hex) 과 비교
"""

import numpy as np

NPY_DIR = r"C:\Users\111eh\AppData\Local\Temp\BNZ.6a153c8824bc5259"
SIM_HEX = r"C:\Users\111eh\INTELLIGENT_SYSTEM_DESIGN\assign4_code\CNN_Accelerator\conv1_out.hex"
IMG_IDX = 0

# ── 1. Python reference 계산 ──────────────────────────────────────────────────
weight = np.load(f"{NPY_DIR}/layer1_0_weight.npy")   # (8, 1, 3, 3) int8
inp    = np.load(f"{NPY_DIR}/input.npy")              # (10000,1,28,28) int8
img    = inp[IMG_IDX, 0].astype(np.int32)             # (28,28)

OC, IC, KH, KW = weight.shape   # 8,1,3,3
OUT_H, OUT_W = 26, 26

ref = np.zeros((OC, OUT_H, OUT_W), dtype=np.int32)
for oc in range(OC):
    for r in range(OUT_H):
        for c in range(OUT_W):
            acc = 0
            for kr in range(KH):
                for kc in range(KW):
                    acc += int(img[r+kr, c+kc]) * int(weight[oc, 0, kr, kc])
            # arithmetic right shift 10 + ReLU + saturate [0,127]
            shifted = acc >> 10
            ref[oc, r, c] = max(0, min(127, shifted))

ref = ref.astype(np.int8)
print(f"[Python] ref shape: {ref.shape}, min={ref.min()}, max={ref.max()}")

# ── 2. 시뮬레이션 결과 로드 ───────────────────────────────────────────────────
raw = []
with open(SIM_HEX, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            val = int(line, 16)
            # 8-bit unsigned → signed
            raw.append(val if val < 128 else val - 256)

raw = np.array(raw, dtype=np.int8).reshape(8, 26, 26)
print(f"[Sim]    sim shape: {raw.shape}, min={raw.min()}, max={raw.max()}")

# ── 3. 비교 ───────────────────────────────────────────────────────────────────
match = np.array_equal(ref, raw)
diff  = ref.astype(np.int32) - raw.astype(np.int32)

print(f"\n=== 결과 {'✅ 일치' if match else '❌ 불일치'} ===")
print(f"  최대 오차: {np.abs(diff).max()}")
print(f"  불일치 픽셀 수: {np.sum(diff != 0)} / {8*26*26}")

if not match:
    idx = np.argwhere(diff != 0)
    print("\n  [처음 5개 불일치]")
    print(f"  {'ch':>3} {'row':>4} {'col':>4}  {'ref':>5}  {'sim':>5}  {'diff':>5}")
    for i in idx[:5]:
        oc, r, c = i
        print(f"  {oc:3d} {r:4d} {c:4d}  {ref[oc,r,c]:5d}  {raw[oc,r,c]:5d}  {diff[oc,r,c]:5d}")


[Python] ref shape: (8, 26, 26), min=0, max=57
[Sim]    sim shape: (8, 26, 26), min=0, max=57

=== 결과 ✅ 일치 ===
  최대 오차: 0
  불일치 픽셀 수: 0 / 5408
